In [1]:
# Montagem do Drive e Instalação de Pendências

from google.colab import drive
drive.mount('/content/drive')

%pip install pydantic-ai google-generativeai --quiet

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.2/101.2 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.5/350.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
# Configuração da API Key

from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')


In [3]:
# Conexão com o Banco de Dados

import sqlite3

DB_PATH = "/content/drive/MyDrive/GenAI Database/banco.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tabelas:", cursor.fetchall())

Tabelas: [('dim_consumidores',), ('dim_produtos',), ('dim_vendedores',), ('fat_avaliacoes_pedidos',), ('fat_itens_pedidos',), ('fat_pedido_total',), ('fat_pedidos',)]


In [5]:
# Exploração do Database - Listagem de tabelas e contagem de linhas:

import sqlite3

DB_PATH = "/content/drive/MyDrive/GenAI Database/banco.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

tabelas = cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()

for (nome,) in tabelas:
    count = cursor.execute(f"SELECT COUNT(*) FROM {nome}").fetchone()[0]
    print(f"{nome}: {count} linhas")

dim_consumidores: 99441 linhas
dim_produtos: 32951 linhas
dim_vendedores: 3095 linhas
fat_avaliacoes_pedidos: 95307 linhas
fat_itens_pedidos: 112650 linhas
fat_pedido_total: 99441 linhas
fat_pedidos: 99441 linhas


In [6]:
# Exploração do Database - Inspeção de colunas e Amostragem:

for (nome,) in tabelas:
    cols = [c[1] for c in cursor.execute(f"PRAGMA table_info({nome})").fetchall()]
    amostra = cursor.execute(f"SELECT * FROM {nome} LIMIT 2").fetchall()
    print(f"\n{'='*60}")
    print(f"Tabela: {nome}")
    print(f"Colunas: {cols}")
    print(f"Amostra: {amostra[0]}")


Tabela: dim_consumidores
Colunas: ['id_consumidor', 'prefixo_cep', 'nome_consumidor', 'cidade', 'estado']
Amostra: ('00012a2ce6f8dcda20d059ce98491703', 6273, 'Dr. Davi Pinto', 'OSASCO', 'SP')

Tabela: dim_produtos
Colunas: ['id_produto', 'nome_produto', 'categoria_produto', 'peso_produto_gramas', 'comprimento_centimetros', 'altura_centimetros', 'largura_centimetros']
Amostra: ('00066f42aeeb9f3007548bb9d3f33c38', 'Loção Corporal Preto', 'perfumaria', 300.0, 20.0, 16.0, 16.0)

Tabela: dim_vendedores
Colunas: ['id_vendedor', 'nome_vendedor', 'prefixo_cep', 'cidade', 'estado']
Amostra: ('0015a82c2db000af6aaaf3ae2ecb0532', 'Amanda Sá', 9080, 'SANTO ANDRE', 'SP')

Tabela: fat_avaliacoes_pedidos
Colunas: ['id_avaliacao', 'id_pedido', 'avaliacao', 'titulo_comentario', 'comentario', 'data_comentario', 'data_resposta']
Amostra: ('7bc2406110b926393aa56f80a40eba40', '73fc7af87114b39712e6da79b0a377eb', 4, 'Sem título', 'Sem comentário', '2018-01-18 00:00:00', '2018-01-18 21:46:59')

Tabela: fat_it

In [7]:
# Exploração do Database - Mapeamento de valores únicos

# Status possíveis de um pedido
status = cursor.execute("SELECT DISTINCT status FROM fat_pedidos").fetchall()
print("Status:", [s[0] for s in status])

# Valores de entrega_no_prazo
prazo = cursor.execute("SELECT DISTINCT entrega_no_prazo FROM fat_pedidos").fetchall()
print("entrega_no_prazo:", [p[0] for p in prazo])

# Todas as categorias de produto
cats = cursor.execute("SELECT DISTINCT categoria_produto FROM dim_produtos").fetchall()
print("Categorias:", [c[0] for c in cats])

# Período dos dados
periodo = cursor.execute("SELECT MIN(data_pedido), MAX(data_pedido) FROM fat_pedido_total").fetchone()
print("Período:", periodo)

# Escala de avaliações
avals = cursor.execute("SELECT MIN(avaliacao), MAX(avaliacao), ROUND(AVG(avaliacao),2) FROM fat_avaliacoes_pedidos").fetchone()
print("Avaliações (min/max/média):", avals)

Status: ['entregue', 'faturado', 'enviado', 'em processamento', 'indisponível', 'cancelado', 'criado', 'aprovado']
entrega_no_prazo: ['Sim', 'Não Entregue', 'Não']
Categorias: ['perfumaria', 'automotivo', 'cama_mesa_banho', 'utilidades_domesticas', 'relogios_presentes', 'cool_stuff', 'consoles_games', 'moveis_decoracao', 'beleza_saude', 'fashion_calcados', 'informatica_acessorios', 'brinquedos', 'pet_shop', 'esporte_lazer', 'ferramentas_jardim', 'moveis_sala', 'malas_acessorios', 'casa_construcao', 'moveis_cozinha_area_de_servico_jantar_e_jardim', 'construcao_ferramentas_construcao', 'moveis_quarto', 'fashion_roupa_masculina', 'construcao_ferramentas_seguranca', 'fashion_bolsas_e_acessorios', 'fraldas_higiene', None, 'telefonia', 'artigos_de_natal', 'papelaria', 'telefonia_fixa', 'livros_interesse_geral', 'eletronicos', 'pc_gamer', 'bebes', 'eletrodomesticos', 'alimentos', 'agro_industria_e_comercio', 'livros_importados', 'construcao_ferramentas_iluminacao', 'bebidas', 'construcao_ferr

In [8]:
# Exploração do Database - Verificação de relacionamento entre as tabelas

# Verificar integridade do join principal
resultado = cursor.execute("""
    SELECT COUNT(*) FROM fat_pedidos fp
    LEFT JOIN dim_consumidores dc ON fp.id_consumidor = dc.id_consumidor
    WHERE dc.id_consumidor IS NULL
""").fetchone()
print(f"Pedidos sem consumidor correspondente: {resultado[0]}")

Pedidos sem consumidor correspondente: 0
